# DEH 30-Day PySpark Challenge
## Day 20 — Spark SQL

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

products_schema = StructType([
    StructField("product_id",     StringType(), True),
    StructField("product_name",   StringType(), True),
    StructField("category",       StringType(), True),
    StructField("sub_category",   StringType(), True),
    StructField("unit_price",     DoubleType(), True),
    StructField("cost_price",     DoubleType(), True),
    StructField("supplier",       StringType(), True),
    StructField("stock_quantity", IntegerType(),True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

## Step 5 — Registering temp views

In [ ]:
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")

print("Temp views registered: orders, customers, products")

## Step 6 — Querying with spark.sql()

In [ ]:
result = spark.sql("""
    SELECT order_id, customer_id, unit_price, status
    FROM orders
    WHERE unit_price > 500
    ORDER BY unit_price DESC
""")

result.show(5)

## Step 7 — Joins and aggregations in SQL

In [ ]:
result = spark.sql("""
    SELECT
        c.segment,
        COUNT(o.order_id) AS order_count,
        ROUND(SUM(o.unit_price), 2) AS total_revenue,
        ROUND(AVG(o.unit_price), 2) AS avg_order_value
    FROM orders o
    INNER JOIN customers c
        ON o.customer_id = c.customer_id
    GROUP BY c.segment
    ORDER BY total_revenue DESC
""")

result.show()

## Step 8 — Window functions in SQL

In [ ]:
result = spark.sql("""
    SELECT region, order_id, unit_price, rnk
    FROM (
        SELECT
            region, order_id, unit_price,
            RANK() OVER (PARTITION BY region ORDER BY unit_price DESC) AS rnk
        FROM orders
    )
    WHERE rnk = 1
""")

result.show()

## Step 9 — External tables

In [ ]:
# First write some Parquet data to query as an external table
orders_df.write.mode("overwrite").parquet(f"s3a://{BUCKET}/output/orders_parquet/")

# Create an external table pointing to that S3 location
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS orders_external
    USING PARQUET
    LOCATION 's3a://{BUCKET}/output/orders_parquet/'
""")

spark.sql("SELECT * FROM orders_external LIMIT 5").show()

In [ ]:
# Dropping the external table removes only metadata — S3 data is untouched
spark.sql("DROP TABLE orders_external")

# Verify data still exists in S3
verify_df = spark.read.parquet(f"s3a://{BUCKET}/output/orders_parquet/")
print(f"Data still in S3 after dropping table: {verify_df.count()} rows")

---
## Assignment — Day 20

---

### Task 1
Register `orders.csv` and `products.csv` as temp views.  
Write a SQL query joining them showing total revenue per `category`, sorted descending.

In [ ]:
# Task 1 — Write your code here


### Task 2
Using SQL, find the top 3 most expensive orders per `region` using `RANK() OVER (PARTITION BY ... ORDER BY ...)`.

In [ ]:
# Task 2 — Write your code here


### Task 3
Write the same query from Task 2 using the PySpark DataFrame API instead of SQL.  
Compare both approaches — which do you find more readable?

In [ ]:
# Task 3 — Write your code here


### Task 4
Create an external table pointing to a Parquet file you wrote earlier in S3.  
Query it with SQL.  
Then drop the table and verify your data in S3 is still intact.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*